In [1]:
import scanpy as sc
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.preprocessing import MinMaxScaler
import scipy.sparse as sp
import anndata as ad
import gseapy as gp
from copy import deepcopy
import scanpy.external as sce
import pickle

# sc.logging.print_version()
sc.settings.verbosity = 3
sc.set_figure_params(figsize=(5,5))

import chronopack as cp

In [2]:
import os
from pathlib import Path

# Folder holding one sub-folder per experiment; see analyses/README.md.
DATA_DIR = Path(os.environ.get("CHRONOPACK_DATA_DIR", "../data"))
EXP_DIR = DATA_DIR / "thp1-k562-lps.hg38.rep"
NEGATIVE_CONTROL_PATH = Path('../inputs/k562_neg-ctrl_mtx-agg.pkl')

# The bulk K562 negative control (N706, ArrayExpress E-MTAB-15966) summed per time tag. If the pickle is missing,
# download N706's DGE table and rebuild it (written to a temporary file first, so a parallel run never reads half of it).
if not NEGATIVE_CONTROL_PATH.exists():
    n706 = Path('../inputs/N706_Bulk_K562_Control_600_Cells.aligned.sorted.merged.tagged.dge.time_tags.csv.gz')
    cp.io.download_arrayexpress_files('E-MTAB-15966', [n706])
    tmp = NEGATIVE_CONTROL_PATH.with_name(f'{NEGATIVE_CONTROL_PATH.name}.{os.getpid()}.tmp')
    cp.io.sum_dge_by_time_tag(n706).to_pickle(tmp)
    tmp.replace(NEGATIVE_CONTROL_PATH)

In [3]:
adata = sc.read_h5ad(EXP_DIR / 'output/adata_full.chk1.h5ad')

# Trajectory clusters 0 and 1 are K562 states; cluster 2 is THP1.
cell_type_by_traj_cluster = {'0': 'K562', '1': 'K562', '2': 'THP1'}
adata.obs['cell_type'] = adata.obs['traj_clusters'].apply(cell_type_by_traj_cluster.get)

In [4]:
adata_k562 = adata[(adata.obs['cell_type'] == 'K562'), :].copy()
adata_k562.var["total_counts"] = np.asarray(adata_k562.layers["counts"].sum(axis=0)).ravel()

low_count_genes_k562 = set(adata_k562.var.index[adata_k562.var["total_counts"] < 10])
print(f"K562 low-count genes to remove before tradeSeq input writing: {len(low_count_genes_k562)}")

K562 low-count genes to remove before tradeSeq input writing: 7253


In [5]:
adata_thp1 = adata[(adata.obs['cell_type'] == 'THP1'), :].copy()
adata_thp1.var["total_counts"] = np.asarray(adata_thp1.layers["counts"].sum(axis=0)).ravel()

low_count_genes_thp1 = set(adata_thp1.var.index[adata_thp1.var["total_counts"] < 10])
print(f"THP1 low-count genes to remove before tradeSeq input writing: {len(low_count_genes_thp1)}")

THP1 low-count genes to remove before tradeSeq input writing: 10695


# Compute correlations

In [6]:
corr_k562 = cp.tl.identify_tag_bias_candidates(
    adata_k562,
    NEGATIVE_CONTROL_PATH,
    method='spline',
    method_kwargs=dict(s=None, k=3),
    local_r_threshold=0.8,
    local_fraction_threshold=0.5,
)

corr_thp1 = cp.tl.identify_tag_bias_candidates(
    adata_thp1,
    NEGATIVE_CONTROL_PATH,
    method='spline',
    method_kwargs=dict(s=None, k=3),
    local_r_threshold=0.7,
    local_fraction_threshold=0.3,
)


In [7]:
corr_k562 = cp.tl.filter_tag_bias_candidates(
    corr_k562,
    local_r_threshold=0.7,
    local_fraction_threshold=0.25,
    use_residual_only=True,
)

print("Tag-bias candidates:", int(corr_k562["tag_bias_candidate"].sum()))

corr_k562[
    [
        "pearson_r_residual",
        "residual_explained_total_fraction",
        "pearson_r_diff",
        "diff_residual_explained_fraction",
        "tag_bias_candidate_residual",
        "tag_bias_candidate_diff",
        "tag_bias_candidate",
    ]
].head(20)

excl_genes_k562 = cp.tl.tag_bias_exclusion_genes(corr_k562)

Tag-bias candidates: 857


In [8]:
corr_thp1 = cp.tl.filter_tag_bias_candidates(
    corr_thp1,
    local_r_threshold=0.7,
    local_fraction_threshold=0.25,
    use_residual_only=True,
)

print("Tag-bias candidates:", int(corr_thp1["tag_bias_candidate"].sum()))

corr_thp1[
    [
        "pearson_r_residual",
        "residual_explained_total_fraction",
        "pearson_r_diff",
        "diff_residual_explained_fraction",
        "tag_bias_candidate_residual",
        "tag_bias_candidate_diff",
        "tag_bias_candidate",
    ]
].head(20)

excl_genes_thp1 = cp.tl.tag_bias_exclusion_genes(corr_thp1)

Tag-bias candidates: 507


# Pre-filter @K562

In [9]:
adata_k562 = cp.io.filter_technical_genes(adata_k562, exclude_genes=excl_genes_k562)
adata_k562 = cp.io.filter_low_count_genes(adata_k562, min_total_counts=10, counts_layer="counts")
adata_k562

AnnData object with n_obs × n_vars = 3128 × 11928
    obs: 'COUNTS', 'sample', 'time', 'time_group', 'n_genes_by_counts', 'total_counts', 'total_counts_mt', 'pct_counts_mt', 'total_counts_rb', 'pct_counts_rb', 'n_genes', 'doublet_score', 'predicted_doublet', 'S_score', 'G2M_score', 'phase', 'leiden_unintegrated', 'THP1_score', 'K562_score', 'THP1_minus_K562', 'cell_type', 'traj_clusters', 'terminal_fate_probs_traj_0', 'terminal_fate_probs_traj_1', 'terminal_fate_probs_traj_2', 'fate_0', 'fate_1', 'fate_2', 'fate_entropy', 'fate_maxprob', 'traj_clusters_mass_ret_cell'
    var: 'mt', 'rb', 'n_cells_by_counts', 'mean_counts', 'pct_dropout_by_counts', 'total_counts', 'n_cells', 'highly_variable', 'means', 'dispersions', 'dispersions_norm'
    uns: 'cell_type_colors', 'chronopack', 'hvg', 'leiden_unintegrated', 'leiden_unintegrated_colors', 'log1p', 'neighbors', 'ot_graph', 'ot_umap', 'pca', 'phase_colors', 'sample_colors', 'scrublet', 'terminal_cell_dist_idx_last', 'terminal_fate_probs_tra

# Pre-filter @THP1

In [10]:
adata_thp1 = cp.io.filter_technical_genes(adata_thp1, exclude_genes=excl_genes_thp1)
adata_thp1 = cp.io.filter_low_count_genes(adata_thp1, min_total_counts=10, counts_layer="counts")
adata_thp1

AnnData object with n_obs × n_vars = 787 × 10117
    obs: 'COUNTS', 'sample', 'time', 'time_group', 'n_genes_by_counts', 'total_counts', 'total_counts_mt', 'pct_counts_mt', 'total_counts_rb', 'pct_counts_rb', 'n_genes', 'doublet_score', 'predicted_doublet', 'S_score', 'G2M_score', 'phase', 'leiden_unintegrated', 'THP1_score', 'K562_score', 'THP1_minus_K562', 'cell_type', 'traj_clusters', 'terminal_fate_probs_traj_0', 'terminal_fate_probs_traj_1', 'terminal_fate_probs_traj_2', 'fate_0', 'fate_1', 'fate_2', 'fate_entropy', 'fate_maxprob', 'traj_clusters_mass_ret_cell'
    var: 'mt', 'rb', 'n_cells_by_counts', 'mean_counts', 'pct_dropout_by_counts', 'total_counts', 'n_cells', 'highly_variable', 'means', 'dispersions', 'dispersions_norm'
    uns: 'cell_type_colors', 'chronopack', 'hvg', 'leiden_unintegrated', 'leiden_unintegrated_colors', 'log1p', 'neighbors', 'ot_graph', 'ot_umap', 'pca', 'phase_colors', 'sample_colors', 'scrublet', 'terminal_cell_dist_idx_last', 'terminal_fate_probs_traj

# Prepare input files for TradeSeq (per cell type)

In [11]:
adata_k562.write_h5ad(EXP_DIR / 'output/adata_k562_prefiltered.fixed.h5ad')
adata_thp1.write_h5ad(EXP_DIR / 'output/adata_thp1_prefiltered.fixed.h5ad')

In [12]:
cp.tl.fit_gene_dynamics(
    adata_k562,
    output_dir=EXP_DIR / 'intmd/sc-K562_to_tradeSeq.fixed',
    prepare_inputs=True,
    run_backend=False,
)
cp.tl.fit_gene_dynamics(
    adata_thp1,
    output_dir=EXP_DIR / 'intmd/sc-THP1_to_tradeSeq.fixed',
    prepare_inputs=True,
    run_backend=False,
)


GeneDynamicsResult(backend='tradeseq', key_added='gene_dynamics', metadata={'time_key': 'time', 'layer': 'counts', 'input_dir': '/new-stg/home/alex/tmp/ChronoPack/verification_runs/prepare_edits_check_20260930_b/data/thp1-k562-lps.hg38.rep/intmd/sc-THP1_to_tradeSeq.fixed', 'n_points': 9, 'workers': 4, 'screen_nknots': 4, 'final_nknots': 8, 'p_threshold': 0.05, 'outputs': {'association': 'asso_df.1trajectory.nknots4.csv', 'significant': 'sig_df.1trajectory.nknots4.csv', 'sce_screen': 'sce_screen.1trajectory.nknots4.rds', 'sce': 'sce.1trajectory.nknots8.rds', 'screen_genes': 'screen_genes.1trajectory.nknots4.csv', 'session_info': 'sessionInfo.tradeSeqFits.txt'}, 'legacy_outputs': {'association': [], 'significant': [], 'sce_screen': [], 'sce': [], 'screen_genes': []}, 'prepared_paths': {'counts': '/new-stg/home/alex/tmp/ChronoPack/verification_runs/prepare_edits_check_20260930_b/data/thp1-k562-lps.hg38.rep/intmd/sc-THP1_to_tradeSeq.fixed/counts_genes_x_cells.csv.gz', 'cell_times': '/new-s